# 🦅 Fine-tuning SmolLM2-1.7B — QLoRA on Colab Free

**Model:** `HuggingFaceTB/SmolLM2-1.7B-Instruct`  
**Method:** QLoRA (4-bit) via Unsloth  
**GPU:** T4 15GB (Colab Free)  
**Training time:** ~5 min (3 epochs)

### Before you start:
1. `Runtime` → `Change runtime type` → **T4 GPU**
2. Run cells one by one

## 1. Installation

In [8]:
%%capture
!pip install unsloth
!pip install trl accelerate peft datasets bitsandbytes

## 2. Check GPU

In [2]:
!nvidia-smi

Fri Apr 17 13:08:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Load model + QLoRA

In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="HuggingFaceTB/SmolLM2-1.7B-Instruct",
    max_seq_length=512,
    load_in_4bit=True,       # QLoRA — saves VRAM
    dtype=None,              # auto
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # rank — higher = more trainable params
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

HuggingFaceTB/SmolLM2-1.7B-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.6 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable params: 18,087,936


## 4. Dataset

Sample data — **replace with your own** in `{instruction, input, output}` format

In [4]:
from datasets import Dataset

# ===== YOUR DATA — edit here =====
data = [
    {
        "instruction": "Answer briefly and clearly.",
        "input": "What is machine learning?",
        "output": "Machine learning is a field of AI where models learn from data without being explicitly programmed."
    },
    {
        "instruction": "Answer briefly and clearly.",
        "input": "What is fine-tuning?",
        "output": "Fine-tuning is adapting a pre-trained model to a specific task by training it further on a small dataset."
    },
    {
        "instruction": "Answer briefly and clearly.",
        "input": "What is LoRA?",
        "output": "LoRA (Low-Rank Adaptation) is a fine-tuning method that trains only small adapters instead of the full model — saving VRAM and time."
    },
    {
        "instruction": "Answer briefly and clearly.",
        "input": "What is a tokenizer?",
        "output": "A tokenizer converts text into numbers (tokens) that a language model can understand."
    },
    {
        "instruction": "Answer briefly and clearly.",
        "input": "What is a GPU?",
        "output": "A GPU is a graphics processor with thousands of small cores — ideal for the parallel matrix operations needed in AI training."
    },
]
# ===== END OF DATA =====

# Format to chat template
def format_prompt(example):
    text = f"""<|im_start|>system
{example['instruction']}<|im_end|>
<|im_start|>user
{example['input']}<|im_end|>
<|im_start|>assistant
{example['output']}<|im_end|>"""
    return {"text": text}

dataset = Dataset.from_list(data).map(format_prompt)
print(f"Dataset size: {len(dataset)} examples")
print("\nExample:")
print(dataset[0]["text"])

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset size: 5 examples

Example:
<|im_start|>system
Answer briefly and clearly.<|im_end|>
<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
Machine learning is a field of AI where models learn from data without being explicitly programmed.<|im_end|>


## 5. Training

In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # effective batch = 2 x 4 = 8
        num_train_epochs=3,             # passes through the dataset
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        output_dir="./output",
        optim="adamw_8bit",
        save_strategy="no",
        report_to="none",
    ),
)

print("Starting training...")
trainer.train()
print("✅ Training complete!")

num_proc must be <= 5. Reducing num_proc to 5 for dataset of size 5.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/5 [00:00<?, ? examples/s]

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5 | Num Epochs = 3 | Total steps = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,087,936 of 1,729,464,320 (1.05% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.295265
2,3.119242
3,2.926611


✅ Training complete!


## 6. Test the fine-tuned model

In [6]:
FastLanguageModel.for_inference(model)

def ask(question, instruction="Answer briefly and clearly."):
    prompt = f"""<|im_start|>system
{instruction}<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("<|im_start|>assistant")[-1].strip()

# Test
print(ask("What is LoRA?"))

Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

system
Answer briefly and clearly.
user
What is LoRA?
assistant
LoRA stands for Long-Term Evolution. It is a wireless broadband communication technology used for mobile broadband internet access, particularly for 4G LTE (Long-Term Evolution) networks. LoRA allows for data transmission rates of up to 100 Mbps and a maximum throughput of 300 Mbps. It was developed to provide faster, more efficient, and more flexible data transmission for mobile devices.


## 7. Save the model (optional)

In [7]:
# Option A: Save only LoRA adapters (~50MB)
model.save_pretrained("smollm2-finetuned-lora")
tokenizer.save_pretrained("smollm2-finetuned-lora")
print("✅ LoRA adapter saved to ./smollm2-finetuned-lora")

# Option B: Merge LoRA into base model and save as GGUF (for mistral.rs / Ollama)
# model.save_pretrained_gguf("smollm2-finetuned", tokenizer, quantization_method="q4_k_m")
# print("✅ GGUF saved")

✅ LoRA adapter saved to ./smollm2-finetuned-lora


## 8. Upload to HuggingFace Hub (optional)

In [ ]:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN")

# model.push_to_hub("your-username/smollm2-finetuned")
# tokenizer.push_to_hub("your-username/smollm2-finetuned")

print("Uncomment the lines above and provide your HuggingFace token to upload the model.")